# LeetCode #1182: Shortest Distance to Target Color

https://leetcode.com/problems/shortest-distance-to-target-color/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n \times q)$ | $O(1)$ |
| **Optimal: Two-Pass Precomputation ★** | $O(n + q)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
For each query `(i, c)`, scan left and right from index `i` to find the nearest occurrence of color `c`. Worst case $O(n)$ per query, $O(n \times q)$ total.

### Optimal: Two-Pass Precomputation ★
For each color $c \in \{1, 2, 3\}$, precompute a `dist[c][i]` array via two linear sweeps — left-to-right (forward nearest) and right-to-left (backward nearest) — then take the element-wise minimum. Each query answers in $O(1)$.

**Why this is better than Brute Force:** Moves all $O(n)$ work to the precomputation phase, reducing per-query cost from $O(n)$ to $O(1)$.

**Constraints:**
* $1 \leq n \leq 5 \times 10^4$
* $1 \leq \text{colors}[i] \leq 3$
* $1 \leq q \leq 5 \times 10^4$
* Queries with no occurrence of color $c$ should return $-1$


## Solutions

### C#

In [ ]:
public class Solution {
    public IList<int> ShortestDistanceColor(int[] colors, int[][] queries) {
        int n = colors.Length;
        // dist[c][i] = shortest distance from index i to any occurrence of color c+1
        int[][] dist = new int[3][];
        for (int c = 0; c < 3; c++) {
            dist[c] = new int[n];
            Array.Fill(dist[c], int.MaxValue);
        }

        // Left-to-right sweep: find the nearest same-color occurrence to the left
        for (int c = 0; c < 3; c++) {
            int last = -1;
            for (int i = 0; i < n; i++) {
                if (colors[i] == c + 1) last = i;
                if (last != -1) dist[c][i] = i - last;
            }
        }

        // Right-to-left sweep: tighten each entry with the nearest right occurrence
        for (int c = 0; c < 3; c++) {
            int last = -1;
            for (int i = n - 1; i >= 0; i--) {
                if (colors[i] == c + 1) last = i;
                if (last != -1 && last - i < dist[c][i])
                    dist[c][i] = last - i;
            }
        }

        // Answer each query in O(1)
        var result = new List<int>();
        foreach (var q in queries)
            result.Add(dist[q[1] - 1][q[0]] == int.MaxValue ? -1 : dist[q[1] - 1][q[0]]);
        return result;
    }
}

### Python

In [ ]:
class Solution:
    def shortestDistanceColor(self, colors: list[int], queries: list[list[int]]) -> list[int]:
        n = len(colors)
        INF = float('inf')
        # dist[c][i] = shortest distance from index i to the nearest cell of color c+1
        dist = [[INF] * n for _ in range(3)]

        # Left sweep: propagate distance from the most recent same-color cell on the left
        for c in range(3):
            last = -1
            for i in range(n):
                if colors[i] == c + 1:
                    last = i
                if last != -1:
                    dist[c][i] = i - last

        # Right sweep: tighten with the nearest same-color cell on the right
        for c in range(3):
            last = -1
            for i in range(n - 1, -1, -1):
                if colors[i] == c + 1:
                    last = i
                if last != -1:
                    dist[c][i] = min(dist[c][i], last - i)

        return [-1 if dist[q[1] - 1][q[0]] == INF else dist[q[1] - 1][q[0]] for q in queries]

### Go

In [ ]:
func shortestDistanceColor(colors []int, queries [][]int) []int {
    n := len(colors)
    const INF = 1<<31 - 1
    // dist[c][i] holds the shortest distance from i to the nearest cell of color c+1
    dist := [3][]int{}
    for c := range dist {
        dist[c] = make([]int, n)
        for i := range dist[c] {
            dist[c][i] = INF
        }
    }

    // Left-to-right pass: nearest same-color cell to the left
    for c := 0; c < 3; c++ {
        last := -1
        for i := 0; i < n; i++ {
            if colors[i] == c+1 {
                last = i
            }
            if last != -1 {
                dist[c][i] = i - last
            }
        }
    }

    // Right-to-left pass: tighten with nearest same-color cell to the right
    for c := 0; c < 3; c++ {
        last := -1
        for i := n - 1; i >= 0; i-- {
            if colors[i] == c+1 {
                last = i
            }
            if last != -1 && last-i < dist[c][i] {
                dist[c][i] = last - i
            }
        }
    }

    result := make([]int, len(queries))
    for k, q := range queries {
        d := dist[q[1]-1][q[0]]
        if d == INF {
            result[k] = -1
        } else {
            result[k] = d
        }
    }
    return result
}

### Rust

In [ ]:
impl Solution {
    pub fn shortest_distance_color(colors: Vec<i32>, queries: Vec<Vec<i32>>) -> Vec<i32> {
        let n = colors.len();
        let inf = i32::MAX;
        // dist[c][i] = closest distance from index i to any cell of color c+1
        let mut dist = vec![vec![inf; n]; 3];

        // Left-to-right: propagate from the last seen same-color cell
        for c in 0..3 {
            let mut last: i32 = -1;
            for i in 0..n {
                if colors[i] == (c + 1) as i32 { last = i as i32; }
                if last != -1 { dist[c][i] = i as i32 - last; }
            }
        }

        // Right-to-left: tighten with the closest cell on the right side
        for c in 0..3 {
            let mut last: i32 = -1;
            for i in (0..n).rev() {
                if colors[i] == (c + 1) as i32 { last = i as i32; }
                if last != -1 {
                    dist[c][i] = dist[c][i].min(last - i as i32);
                }
            }
        }

        queries.iter().map(|q| {
            let d = dist[(q[1] - 1) as usize][q[0] as usize];
            if d == inf { -1 } else { d }
        }).collect()
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `colors = [1,1,2,1,3,2,2,3,3]`, `queries = [[1,3],[2,2],[6,1]]`
Color-3 positions: 4,7,8. From index 1, nearest color-3 is at index 4, distance **3**. Color-2 at indices 2,5,6. From index 2, distance **0**. Color-1 at indices 0,1,3. From index 6, nearest is index 3, distance **3**.

### 2. Slightly Complex
**Input:** `colors = [1,2,3]`, `queries = [[0,3]]`
Color-3 only at index 2. Left sweep leaves `dist[2][0] = INF`. Right sweep sets `dist[2][0] = 2`. Answer: **2**.

### 3. Edge Case: Time Factor
**Input:** $n = q = 5 \times 10^4$, all queries ask `(0, 2)` with color-2 only at index $n-1$
Precomputation: 6 sweeps each $O(n)$. Query answering: $q$ lookups each $O(1)$. Total $O(n + q)$ — maximum-size validation.

### 4. Edge Case: Space Factor
**Input:** `colors = [1,1,...,1]` ($n = 5 \times 10^4$, all color 1), `queries = [(i, 2) for all i]`
Colors 2 and 3 never appear. Both sweeps leave `dist[1]` and `dist[2]` filled with INF. Every query returns **-1**. Space remains $O(n)$ — three precomputed arrays.

### 5. Almost-Impossible but Plausible
**Input:** `colors = [1,2,1,2,1]`, `queries = [[2,2]]`
Color-2 at indices 1 and 3, equidistant from index 2 (distance 1 each). Left sweep sets `dist[1][2] = 1`; right sweep confirms minimum stays 1. Answer: **1**. Tests that equidistant cells on both sides produce the correct minimum without double-counting.
